# 自动上下文压缩

长期运行的 Agent 任务经常会超过上下文限制。工具密集型工作流或长对话会快速消耗 token 上下文窗口。在 [Effective Context Engineering for AI Agents](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents) 中，我们讨论了如何管理上下文以帮助避免性能下降和上下文腐烂。

Claude Agent Python SDK 可以通过在 token 使用量超过可配置阈值时自动压缩对话历史来帮助管理此上下文，从而允许任务继续超过典型的 200k token 上下文限制。

在本 cookbook 中，我们将通过**客户服务工作流**来演示上下文压缩。想象你已经构建了一个 AI 客户服务 Agent，负责处理支持工单队列。对于每个工单，你必须分类问题、搜索知识库、设置优先级、路由到适当的团队、起草响应并标记为完成。当你处理一个又一个工单时，对话历史会充满分类、知识库搜索和起草的响应——快速消耗数千个 token。

## 什么是上下文压缩？

在使用工具的 Agent 工作流中，随着 Agent 对复杂任务进行迭代，对话可能会变得非常大。`compaction_control` 参数通过以下方式提供自动上下文管理：

1. 监控对话中每轮的 token 使用量
2. 当超过阈值时，作为用户轮次注入摘要提示
3. 让模型生成包裹在 `<summary></summary>` 标签中的摘要。这些标签不会被解析，但可以帮助指导模型。
4. 清除对话历史并仅使用摘要恢复
5. 使用压缩的上下文继续任务

## 在本 cookbook 结束时，你将能够：
 
 - 理解如何在迭代工作流中有效管理上下文限制
 - 编写利用自动上下文压缩的 Agent
 - 设计跨多次迭代保持专注的工作流

## 前置条件

在遵循本指南之前，确保你有：

**必备知识**

- 对 Agent 模式和工具调用的基本理解

**必备工具**

- Python 3.11 或更高版本
- Anthropic API 密钥
- Anthropic SDK >= 0.74.1

> **使用 Opus 4.6？** 我们建议使用[服务端压缩](https://docs.anthropic.com/en/docs/build-with-claude/compaction)，它会自动处理上下文管理，无需任何 SDK 级别的配置。
>
> 本 cookbook 涵盖**基于 SDK 的压缩**，如果你使用旧模型或想要使用不同的（更便宜的）模型进行摘要，这很有用。

## 设置

首先，安装所需的依赖：

In [1]:
# %pip install -qU anthropic python-dotenv

注意：确保你的 .env 文件包含：

`ANTHROPIC_API_KEY=your_key_here`

加载你的环境变量并配置客户端。我们还加载了一个辅助工具来可视化 Claude 消息响应。


In [2]:
from dotenv import load_dotenv

load_dotenv()

MODEL = "claude-sonnet-4-6"

## 设置场景

在 [utils/customer_service_tools.py](utils/customer_service_tools.py) 中，我们定义了几个用于处理客户支持工单的函数：

- `get_next_ticket()` - 从队列中检索下一个未处理的工单
- `classify_ticket(ticket_id, category)` - 将问题分类为账单、技术、账户、产品或运输
- `search_knowledge_base(query)` - 查找相关的帮助文章和解决方案
- `set_priority(ticket_id, priority)` - 分配优先级（低、中、高、紧急）
- `route_to_team(ticket_id, team)` - 将工单路由到适当的支持团队
- `draft_response(ticket_id, response_text)` - 创建面向客户的响应
- `mark_complete(ticket_id)` - 完成已处理的工单

对于客户服务 Agent，这些工具可以系统地处理工单。每个工单都需要分类、研究、优先级设置、路由和响应起草。当按顺序处理 20-30 个工单时，对话历史会充满每个分类、每个知识库搜索和每个起草响应的工具结果，导致 token 线性增长。

工具上使用 `beta_tool` 装饰器使其对 Claude Agent 可访问。装饰器提取函数参数和文档字符串，并将这些作为工具元数据提供给 Claude。

```python
import anthropic
from anthropic import beta_tool

@beta_tool
def get_next_ticket() -> dict:
    """从队列中检索下一个未处理的支持工单。"""
    ...
```


In [3]:
import anthropic
from utils.customer_service_tools import (
    classify_ticket,
    draft_response,
    get_next_ticket,
    initialize_ticket_queue,
    mark_complete,
    route_to_team,
    search_knowledge_base,
    set_priority,
)

client = anthropic.Anthropic()

tools = [
    get_next_ticket,
    classify_ticket,
    search_knowledge_base,
    set_priority,
    route_to_team,
    draft_response,
    mark_complete,
]

## 基准：不使用压缩运行

让我们从一个现实的客户服务场景开始：处理支持工单队列。

工作流如下：

**对于每个工单：**
1. 使用 `get_next_ticket()` 获取工单
2. 分类问题类别（账单、技术、账户、产品、运输）
3. 搜索知识库获取相关信息
4. 设置适当的优先级（低、中、高、紧急）
5. 路由到正确的团队
6. 起草客户响应
7. 标记工单完成
8. 移动到下一个工单

**挑战**：队列中有 5 个工单，每个需要 7 次工具调用，Claude 将进行 35 次或更多次工具调用。每个步骤的结果（包括分类、知识库搜索和起草的响应）会累积在对话历史中。没有压缩，所有这些数据都会保留在每个工单的内存中，到第 5 个工单时，上下文包含前 4 个工单的所有完整详细信息。

让我们首先**不使用压缩**运行此工作流并观察发生的情况：

In [4]:
from anthropic.types.beta import BetaMessageParam

num_tickets = 5
initialize_ticket_queue(num_tickets)

messages: list[BetaMessageParam] = [
    {
        "role": "user",
        "content": f"""You are an AI customer service agent. Your task is to process support tickets from a queue.

For EACH ticket, you must complete ALL these steps:

1. **Fetch ticket**: Call get_next_ticket() to retrieve the next unprocessed ticket
2. **Classify**: Call classify_ticket() to categorize the issue (billing/technical/account/product/shipping)
3. **Research**: Call search_knowledge_base() to find relevant information for this ticket type
4. **Prioritize**: Call set_priority() to assign priority (low/medium/high/urgent) based on severity
5. **Route**: Call route_to_team() to assign to the appropriate team
6. **Draft**: Call draft_response() to create a helpful customer response using KB information
7. **Complete**: Call mark_complete() to finalize this ticket
8. **Continue**: Immediately fetch the next ticket and repeat

IMPORTANT RULES:
- Process tickets ONE AT A TIME in sequence
- Complete ALL 7 steps for each ticket before moving to the next
- Keep fetching and processing tickets until you get an error that the queue is empty
- There are {num_tickets} tickets total - process all of them
- Be thorough but efficient

Begin by fetching the first ticket.""",
    }
]

total_input = 0
total_output = 0
turn_count = 0

runner = client.beta.messages.tool_runner(
    model=MODEL,
    max_tokens=4096,
    tools=tools,
    messages=messages,
)

for message in runner:
    messages_list = list(runner._params["messages"])
    turn_count += 1
    total_input += message.usage.input_tokens
    total_output += message.usage.output_tokens
    print(
        f"Turn {turn_count:2d}: Input={message.usage.input_tokens:7,} tokens | "
        f"Output={message.usage.output_tokens:5,} tokens | "
        f"Messages={len(messages_list):2d} | "
        f"Cumulative In={total_input:8,}"
    )

print(f"\n{'=' * 60}")
print("BASELINE RESULTS (NO COMPACTION)")
print(f"{'=' * 60}")
print(f"Total turns:   {turn_count}")
print(f"Input tokens:  {total_input:,}")
print(f"Output tokens: {total_output:,}")
print(f"Total tokens:  {total_input + total_output:,}")
print(f"{'=' * 60}")

Turn  1: Input=  1,537 tokens | Output=   57 tokens | Messages= 1 | Cumulative In=   1,537
Turn  2: Input=  1,760 tokens | Output=  102 tokens | Messages= 3 | Cumulative In=   3,297
Turn  3: Input=  1,905 tokens | Output=   88 tokens | Messages= 5 | Cumulative In=   5,202
Turn  4: Input=  2,237 tokens | Output=   84 tokens | Messages= 7 | Cumulative In=   7,439
Turn  5: Input=  2,385 tokens | Output=   89 tokens | Messages= 9 | Cumulative In=   9,824
Turn  6: Input=  2,537 tokens | Output=  301 tokens | Messages=11 | Cumulative In=  12,361
Turn  7: Input=  2,888 tokens | Output=   67 tokens | Messages=13 | Cumulative In=  15,249
Turn  8: Input=  3,079 tokens | Output=   56 tokens | Messages=15 | Cumulative In=  18,328
Turn  9: Input=  3,316 tokens | Output=   91 tokens | Messages=17 | Cumulative In=  21,644
Turn 10: Input=  3,450 tokens | Output=   84 tokens | Messages=19 | Cumulative In=  25,094
Turn 11: Input=  3,777 tokens | Output=   84 tokens | Messages=21 | Cumulative In=  28,871

现在我们有了基准，我们更好地了解了上下文如何在没有压缩的情况下增长。如你所见，每轮都会导致 token 线性增长，因为每轮都会向输入添加更多 token。

这导致高 token 消耗和可能快速达到上下文限制。到第 27 轮，仅 5 个工单我们就累积了 150,000 个输入 token。

让我们查看 Claude 在不使用压缩处理所有 5 个工单后的最终响应：

In [5]:
print(message.content[-1].text)

---

## ✅ ALL TICKETS PROCESSED SUCCESSFULLY!

**Summary of Completed Work:**

I have successfully processed all 5 tickets from the queue. Here's what was accomplished:

1. **TICKET-1** - Sam Smith - Payment method update error
   - Category: Billing | Priority: High | Team: billing-team
   
2. **TICKET-2** - Morgan Johnson - Missing delivery
   - Category: Shipping | Priority: High | Team: logistics-team
   
3. **TICKET-3** - Morgan Jones - Email address change request
   - Category: Account | Priority: Medium | Team: account-services
   
4. **TICKET-4** - Alex Johnson - Wrong item delivered
   - Category: Shipping | Priority: High | Team: logistics-team
   
5. **TICKET-5** - Morgan Jones - Refund request for cancelled subscription
   - Category: Billing | Priority: High | Team: billing-team

Each ticket was:
✅ Classified correctly
✅ Researched in the knowledge base
✅ Assigned appropriate priority
✅ Routed to the correct team
✅ Given a detailed, helpful customer response
✅ Marked as c

### 理解问题

在上面的基准工作流中，Claude 必须：
- 按顺序处理 **5 个支持工单**
- 每个工单完成 **7 个步骤**（获取、分类、研究、优先级设置、路由、起草、完成）
- 进行 **35 次工具调用**，结果累积在对话历史中
- 在内存中存储**每个分类、每个知识库搜索、每个起草的响应**

**为什么会发生这种情况**：
1. **Token 线性增长** - 每次工具使用时，整个对话历史（包括所有以前的工具结果）都会发送给 Claude
2. **上下文污染** - 处理工单 B 时，工单 A 的分类和起草的响应仍保留在上下文中
3. **复合成本** - 当你处理到第 5 个工单时，每次 API 调用都会发送前 4 个工单的所有数据
4. **响应变慢** - 处理大量上下文需要更长时间
5. **达到限制的风险** - 最终你会达到 200k token 上下文窗口


**我们实际需要的**：完成工单 A 后，我们只需要一个**简要摘要**（工单已解决、类别、优先级）——而不是完整的分类结果、知识库搜索和完整的起草响应。详细的工作流应该被丢弃，只保留完成摘要。

让我们看看自动上下文压缩如何解决这个问题。

## 启用自动上下文压缩

让我们运行完全相同的客户服务工作流，但启用自动上下文压缩。我们只需向工具运行器添加 `compaction_control` 参数。

`compaction_control` 参数有一个必需字段和几个可选字段：

- **`enabled`**（必需）：布尔值，用于打开/关闭压缩
- **`context_token_threshold`**（可选）：触发压缩的 token 数量（默认：100,000）
- **`model`**（可选）：用于摘要的模型（默认为主模型）
- **`summary_prompt`**（可选）：用于生成摘要的自定义提示

对于此客户服务工作流，我们将使用 **5,000 token 阈值**。这意味着处理多个工单后压缩将自动触发。这允许 Claude：
1. **保留完成摘要**（已解决的工单、类别、结果）
2. **丢弃详细的工具结果**（完整的知识库文章、完整的分类、起草的响应文本）
3. 在处理下一批工单时**重新开始**

这模拟了真实支持 Agent 的工作方式：解决工单、简要记录、移至下一个案例。

In [6]:
# Re-initialize queue and run with compaction
initialize_ticket_queue(num_tickets)

total_input_compact = 0
total_output_compact = 0
turn_count_compact = 0
compaction_count = 0
prev_msg_count = 0

runner = client.beta.messages.tool_runner(
    model=MODEL,
    max_tokens=4096,
    tools=tools,
    messages=messages,
    compaction_control={
        "enabled": True,
        "context_token_threshold": 5000,
    },
)

for message in runner:
    turn_count_compact += 1
    total_input_compact += message.usage.input_tokens
    total_output_compact += message.usage.output_tokens
    messages_list = list(runner._params["messages"])
    curr_msg_count = len(messages_list)

    if curr_msg_count < prev_msg_count:
        # We can identify compaction when the message count decreases
        compaction_count += 1

        print(f"\n{'=' * 60}")
        print(f"🔄 Compaction occurred! Messages: {prev_msg_count} → {curr_msg_count}")
        print("   Summary message after compaction:")
        print(messages_list[-1]["content"][-1].text)  # type: ignore
        print(f"\n{'=' * 60}")

    prev_msg_count = curr_msg_count
    print(
        f"Turn {turn_count_compact:2d}: Input={message.usage.input_tokens:7,} tokens | "
        f"Output={message.usage.output_tokens:5,} tokens | "
        f"Messages={len(messages_list):2d} | "
        f"Cumulative In={total_input_compact:8,}"
    )

print(f"\n{'=' * 60}")
print("OPTIMIZED RESULTS (WITH COMPACTION)")
print(f"{'=' * 60}")
print(f"Total turns:   {turn_count_compact}")
print(f"Compactions:   {compaction_count}")
print(f"Input tokens:  {total_input_compact:,}")
print(f"Output tokens: {total_output_compact:,}")
print(f"Total tokens:  {total_input_compact + total_output_compact:,}")
print(f"{'=' * 60}")

Turn  1: Input=  1,537 tokens | Output=   57 tokens | Messages= 1 | Cumulative In=   1,537
Turn  2: Input=  1,755 tokens | Output=  108 tokens | Messages= 3 | Cumulative In=   3,292
Turn  3: Input=  1,906 tokens | Output=   88 tokens | Messages= 5 | Cumulative In=   5,198
Turn  4: Input=  2,216 tokens | Output=   84 tokens | Messages= 7 | Cumulative In=   7,414
Turn  5: Input=  2,364 tokens | Output=   89 tokens | Messages= 9 | Cumulative In=   9,778
Turn  6: Input=  2,516 tokens | Output=  332 tokens | Messages=11 | Cumulative In=  12,294
Turn  7: Input=  2,898 tokens | Output=   67 tokens | Messages=13 | Cumulative In=  15,192
Turn  8: Input=  3,090 tokens | Output=   56 tokens | Messages=15 | Cumulative In=  18,282
Turn  9: Input=  3,325 tokens | Output=   97 tokens | Messages=17 | Cumulative In=  21,607
Turn 10: Input=  3,465 tokens | Output=   90 tokens | Messages=19 | Cumulative In=  25,072
Turn 11: Input=  3,801 tokens | Output=   84 tokens | Messages=21 | Cumulative In=  28,873

启用自动上下文压缩后，我们可以看到每轮的 token 使用量不会线性增长，而是在每次压缩事件后减少。在处理工单期间发生了两次压缩事件，随后的轮次显示总 token 使用量减少。

与基准版本相比，我们只使用了 79,000 个 token。我们还打印了每次压缩事件后生成的摘要消息，显示 Claude 如何有效地将之前的工单详细信息压缩成摘要。

让我们查看启用压缩后处理所有 5 个工单的最终响应。

In [7]:
print(message.content[-1].text)

Perfect! **ALL 5 TICKETS HAVE BEEN SUCCESSFULLY COMPLETED!** 🎉

## Final Summary - All Tickets Processed

### TICKET-5 (Morgan Brown) - **COMPLETED** ✓
- **Issue**: Damaged package (Order #ORD-43312), broken product inside, needs replacement
- **Category**: shipping
- **Priority**: high
- **Team**: logistics-team
- **Status**: resolved
- **Response**: Apologized for damaged shipment, escalated to Logistics Team with HIGH priority, explained they'll process immediate replacement, provide return instructions, and contact customer with tracking and timeline

---

## 🎯 ALL 5 TICKETS COMPLETED

1. ✅ **TICKET-1** (Chris Davis) - Account locked → account-services
2. ✅ **TICKET-2** (Chris Williams) - Billing charge → billing-team  
3. ✅ **TICKET-3** (John Jones) - Google Sheets integration → product-success
4. ✅ **TICKET-4** (Sam Johnson) - Plan comparison → product-success
5. ✅ **TICKET-5** (Morgan Brown) - Damaged shipment → logistics-team

### Processing Statistics
- **Total tickets process

### 比较结果

启用压缩后，我们可以在两次运行中看到 token 节省方面的明显差异，同时保持工作流和最终摘要的质量。

以下是自动上下文压缩带来的变化：

1. **处理多个工单后上下文重置** - 当处理 5-7 个工单产生 5k+ token 的工具结果时，SDK 会自动：
   - 注入摘要提示
   - 让 Claude 生成包裹在 `<summary></summary>` 标签中的完成摘要
   - 清除对话历史并丢弃详细的分类、知识库搜索和响应
   - 仅使用完成摘要继续

2. **输入 token 保持有界** - 随着我们处理更多工单，输入 token 不会累积到 100k+，而是在每次压缩后重置。处理第 5 个工单时，我们不会携带第 1-4 个工单的完整工具结果。

3. **任务成功完成** - 工作流顺利完成所有工单，没有达到上下文限制

4. **质量得以保留** - 摘要保留关键信息：
   - 已处理的工单及其 ID
   - 分配的类别和优先级
   - 路由到的团队
   - 总体进度状态
   
   所有工单仍然得到正确的分类、优先级设置、路由和响应。

5. **自然的工作流** - 这反映了真实支持 Agent 的工作方式：解决工单、在系统中简要记录、关闭它、移至下一个。在处理新工单时，你不会保持每个知识库文章和完整响应草稿处于打开状态。

让我们可视化 token 节省：

In [8]:
# Compare baseline vs compaction
print("=" * 70)
print("TOKEN USAGE COMPARISON")
print("=" * 70)
print(f"{'Metric':<30} {'Baseline':<20} {'With Compaction':<20}")
print("-" * 70)
print(f"{'Input tokens:':<30} {total_input:>19,} {total_input_compact:>19,}")
print(f"{'Output tokens:':<30} {total_output:>19,} {total_output_compact:>19,}")
print(
    f"{'Total tokens:':<30} {total_input + total_output:>19,} {total_input_compact + total_output_compact:>19,}"
)
print(f"{'Compactions:':<30} {'N/A':>19} {compaction_count:>19}")
print("=" * 70)

# Calculate savings
token_savings = (total_input + total_output) - (total_input_compact + total_output_compact)
savings_percent = (
    (token_savings / (total_input + total_output)) * 100 if (total_input + total_output) > 0 else 0
)

print(f"\n💰 Token Savings: {token_savings:,} tokens ({savings_percent:.1f}% reduction)")

TOKEN USAGE COMPARISON
Metric                         Baseline             With Compaction     
----------------------------------------------------------------------
Input tokens:                              204,416              82,171
Output tokens:                               4,422               4,275
Total tokens:                              208,838              86,446
Compactions:                                   N/A                   2

💰 Token Savings: 122,392 tokens (58.6% reduction)


## How Compaction Works Under the Hood

When the `tool_runner` detects that token usage has exceeded the threshold, it automatically:

1. **Pauses the workflow** before making the next API call
2. **Injects a summary request** as a user message asking Claude to summarize progress
3. **Generates a summary** - Claude produces a summary wrapped in `<summary></summary>` tags containing:
   - **Completed tickets**: Brief records of tickets resolved (IDs, categories, priorities, outcomes)
   - **Progress status**: How many tickets processed, how many remain
   - **Key patterns**: Any notable trends across tickets
   - **Next steps**: What to do next (continue processing remaining tickets)
4. **Clears history** - The entire conversation history (including all tool results) is replaced with just the summary
5. **Resumes processing** - Claude continues working with the compressed context, processing the next batch of tickets

## Customizing Compaction Configuration

You can customize how compaction works to fit your specific use case. Here are the key configuration options:

### Adjusting the Threshold

The `context_token_threshold` determines when compaction triggers:

```python
compaction_control={
    "enabled": True,
    "context_token_threshold": 5000,  # Compact after processing 5-7 tickets
}
```

The threshold should not be set too low, otherwise the summary itself could trigger a compaction. We set a threshold of 5,000 tokens for demonstration purposes, but in practice, experiment with different settings to find what works best for your workflow.

Here some general guidelines:

- **Low thresholds (5k-20k)**: 
  - Use for iterative task processing with clear boundaries
  - More frequent compaction, minimal context accumulation
  - Best for sequential entity processing
  
- **Medium thresholds (50k-100k)**: 
  - Multi-phase workflows with fewer, larger natural checkpoints
  - Balance between context retention and management
  - Suitable for workflows with expensive tool calls
  
- **High thresholds (100k-150k)**: 
  - Tasks requiring substantial historical context
  - Less frequent compaction preserves more raw details
  - Higher per-call costs but fewer compactions
  
- **Default (100k)**: Good balance for general long-running tasks

**For ticket processing**: The 5k threshold works well because each ticket's workflow generates substantial tool results, but tickets are independent. After resolving Ticket A, you don't need its detailed KB searches when processing Ticket B.

### Using a Different Model for Summarization

You can also use a faster/cheaper model for generating summaries:

```python
compaction_control={
    "enabled": True,
    "model": "claude-haiku-4-5",  # Use Haiku for cost-effective summaries
}
```

### Custom Summary Prompts

You can provide a custom prompt to guide how summaries are generated. This is especially useful for customer service workflows where you need to preserve specific types of information.

For example, we could define a custom prompt based on our requirements:
- **Ticket summaries** for all completed tickets
- **Categories and priorities** assigned
- **Teams routed to**
- **Progress status** (tickets completed, tickets remaining)
- **Next steps** in the workflow

```python
compaction_control={
    "enabled": True,
    "summary_prompt": """You are processing customer support tickets from a queue.

Create a focused summary that preserves:

1. **COMPLETED TICKETS**: For each ticket you've fully processed:
   - Ticket ID and customer name
   - Issue category and priority assigned
   - Team routed to
   - Brief outcome

2. **PROGRESS STATUS**: 
   - How many tickets you've completed
   - Approximately how many remain in the queue

3. **NEXT STEPS**: Continue processing the next ticket

Format with clear sections and wrap in <summary></summary> tags."""
}
```

## Compaction Without Tools: Simple Chat Loop

While the examples above focus on tool-heavy agentic workflows, context compaction is also valuable for **simple conversational applications** where users drive the conversation.

 **Note:** The `compaction_control` parameter demonstrated above works with `tool_runner` for agentic workflows with tools. For simple chat applications without tools, you'll implement compaction manually using the same principles.

Consider a chat application where users are having extended conversations with Claude—discussing complex topics, iterating on ideas, or working through problems. As the conversation grows, you face the same context accumulation challenges.

**The Difference**: Instead of tool use triggering token growth, it's the back-and-forth conversation itself. Each exchange adds messages to the history:
- User asks a question
- Claude provides a detailed response
- User asks for clarification or elaboration
- Claude responds with more context
- This repeats dozens or hundreds of times

Without compaction, by turn 50 you're sending the entire conversation history (all 50 exchanges) on every API call.

**The Solution**: Implement compaction manually in your chat loop using the same pattern:
1. Track token usage after each turn
2. When threshold is exceeded, request a summary
3. Replace conversation history with the summary
4. Continue the conversation with compressed context

Let's see how to implement this:

In [ ]:
#!/usr/bin/env python3
"""
Simple Compaction Example - User-Driven Chat Loop

This shows the basic pattern for a chat application with compaction.
No tools required - just a simple loop where the user drives continuation.
"""

# Configuration
COMPACTION_THRESHOLD = 3000  # Compact when tokens exceed this (low for demo purposes)

# Structured summarization prompt for compaction
SUMMARY_PROMPT = """You have been working on the task described above but have not yet completed it. Write a continuation summary that will allow you (or another instance of yourself) to resume work efficiently in a future context window where the conversation history will be replaced with this summary. Your summary should be structured, concise, and actionable. Include:

1. **Task Overview**
   - The user's core request and success criteria
   - Any clarifications or constraints they specified

2. **Current State**
   - What has been completed so far
   - Files created, modified, or analyzed (with paths if relevant)
   - Key outputs or artifacts produced

3. **Important Discoveries**
   - Technical constraints or requirements uncovered
   - Decisions made and their rationale
   - Errors encountered and how they were resolved
   - What approaches were tried that didn't work (and why)

4. **Next Steps**
   - Specific actions needed to complete the task
   - Any blockers or open questions to resolve
   - Priority order if multiple steps remain

5. **Context to Preserve**
   - User preferences or style requirements
   - Domain-specific details that aren't obvious
   - Any promises made to the user

Be concise but complete—err on the side of including information that would prevent duplicate work or repeated mistakes.
 Write in a way that enables immediate resumption of the task.

Wrap your summary in <summary></summary> tags."""

# Message history
messages = []

print("Chat with Claude (type 'quit' to exit, or just hit Enter to continue)")
print("This is a demonstration - try having a conversation and watch compaction trigger")
print("=" * 60)

# Simulate a conversation for demo purposes
demo_messages = [
    "Help me understand how Python decorators work",
    "Can you show me an example with a timing decorator?",
    "How would I make a decorator that takes arguments?",
]

for user_input in demo_messages:
    print(f"\nYou: {user_input}")

    # Add user message
    messages.append({"role": "user", "content": user_input})

    # Get Claude's response
    response = client.messages.create(
        model=MODEL,
        max_tokens=2048,
        messages=messages,
    )

    messages.append(
        {
            "role": "assistant",
            "content": response.content,
        }
    )

    print("\nClaude: ", end="")
    for block in response.content:
        if block.type == "text":
            print(f"{block.text[:300]} ...")

    # Check if we should compact
    usage = response.usage

    # Calculate total tokens (includes cache tokens)
    total_input_tokens = (
        usage.input_tokens
        + (usage.cache_creation_input_tokens or 0)
        + (usage.cache_read_input_tokens or 0)
    )
    total_tokens = total_input_tokens + usage.output_tokens

    cache_info = ""
    if usage.cache_creation_input_tokens or usage.cache_read_input_tokens:
        cache_info = f" (cache: {usage.cache_creation_input_tokens or 0} write + {usage.cache_read_input_tokens or 0} read)"

    print(
        f"\n[Tokens: {total_input_tokens} in{cache_info} + {usage.output_tokens} out = {total_tokens} total]"
    )

    if total_tokens > COMPACTION_THRESHOLD:
        print(f"\n{'=' * 60}")
        print(f"🔄 Compacting conversation... {len(messages)} messages → ", end="", flush=True)

        # Get summary using structured prompt
        summary_response = client.messages.create(
            model=MODEL,
            max_tokens=4096,
            messages=messages + [{"role": "user", "content": SUMMARY_PROMPT}],
        )

        summary_text = "".join(
            block.text for block in summary_response.content if block.type == "text"
        )

        # Replace history with summary
        messages = [{"role": "user", "content": summary_text}]

        print("1 message")
        print(f"{'=' * 60}\n")

print(f"Final conversation messages: {messages[-1].get('content')}")

print("\nDemo complete! In a real application, this loop would continue with user input.")

Chat with Claude (type 'quit' to exit, or just hit Enter to continue)
This is a demonstration - try having a conversation and watch compaction trigger

You: Help me understand how Python decorators work


### Understanding the Chat Loop Pattern

The example above demonstrates manual compaction in a conversational context. Here's how it works:

**Key Components**:

1. **Token Tracking**: After each response, calculate total tokens (input + output + cache tokens)
2. **Threshold Check**: When total exceeds threshold, trigger compaction
3. **Summary Request**: Send the same structured SUMMARY_PROMPT to Claude
4. **History Replacement**: Replace entire message history with just the summary
5. **Continue**: Next user message builds on the summary, not full history

**When to Use This Pattern**:

- **Extended brainstorming sessions**: Users exploring ideas with Claude over many turns
- **Learning conversations**: Tutorials or explanations that span dozens of exchanges
- **Iterative refinement**: Users providing feedback on drafts, designs, or solutions
- **Chat applications**: Any multi-turn conversation interface

**Key Differences from Tool Runner**:

| Aspect | Tool Runner (Automatic) | Chat Loop (Manual) |
|--------|------------------------|-------------------|
| **Trigger** | Automatic when threshold reached | You implement threshold check |
| **Summary** | SDK handles summary request | You make explicit API call |
| **History Management** | SDK replaces messages | You manually replace list |
| **Use Case** | Agentic workflows with tools | User-driven conversations |

**Production Considerations**:

1. **Adjust threshold**: Use larger thresholds for real applications
2. **Customize summary prompt**: Tailor to your conversation type (brainstorming vs. technical support vs. tutoring)
3. **Show user indicators**: Display a message like "Summarizing conversation..." so users understand the pause
4. **Preserve key context**: Ensure the summary prompt captures domain-specific information your users care about

This pattern gives you full control over when and how compaction happens, making it ideal for conversational applications where the SDK's automatic tool-runner compaction isn't available.

## Limitations and Considerations

While automatic context compaction is powerful, there are important limitations to understand:

### Server-Side Sampling Loops

**Current Limitation**: Compaction does not work optimally with server-side sampling loops, such as server-side web search tools.

**Why**: Cache tokens accumulate across sampling loops, which can trigger compaction prematurely based on cached content rather than actual conversation history.

This feature works best with:
- ✅ Client-side tools (like the customer service API in this cookbook)
- ✅ Standard agentic workflows with regular tool use
- ✅ File operations, database queries, API calls
- ❌ Server-side Extended Thinking
- ❌ Server-side web search tools

### Information Loss

**Trade-off**: Summaries inherently lose some information. While Claude is good at identifying key points, some details will be compressed or omitted.

**In ticket processing**: 
- ✅ **Retained**: Ticket IDs, categories, priorities, teams, outcomes, progress status
- ❌ **Lost**: Full knowledge base article text, complete drafted response text, detailed classification reasoning

This is usually acceptable, you don't need every KB article and full response text in perpetuity, just the completion records.

**Mitigation**:
- Use custom summary prompts to preserve critical information
- Set higher thresholds for tasks requiring extensive historical context
- Structure your tasks to be modular (each phase builds on summaries, not raw details)

### When NOT to Use Compaction

Avoid compaction for:

1. **Short tasks**: If your task completes within 50k-100k tokens, compaction adds unnecessary overhead
2. **Tasks requiring full audit trails**: Some tasks need access to ALL previous details
3. **Server-side sampling workflows**: As mentioned above, wait for this limitation to be addressed
4. **Highly iterative refinement**: Tasks where each step critically depends on exact details from all previous steps

### When TO Use Compaction

Compaction is ideal for:

1. **Sequential processing**: Like our ticket workflow—process multiple items one after another
2. **Multi-phase workflows**: Where each phase can summarize progress before moving on
3. **Iterative data processing**: Processing large datasets in chunks or entities one at a time
4. **Extended analysis sessions**: Analyzing data across many entities
5. **Batch operations**: Processing hundreds of items where each is independent

**Ticket processing is a perfect use case** because:
- Each ticket workflow is largely independent
- You need completion summaries, not full tool results
- Natural compaction points exist (after completing several tickets)
- The workflow is iterative and sequential

## Summary

Automatic context compaction is a powerful feature that enables long-running agentic workflows to exceed typical context limits. In this cookbook, we've explored compaction through a customer service ticket processing workflow.

### Next Steps

Try implementing compaction in your own workflows:
1. Identify natural compaction points (after processing each item, completing each phase, etc.)
2. Start with an aggressive threshold (5k-10k) if you have clear per-item boundaries
3. Use custom summary prompts to preserve critical information
4. Monitor when compaction triggers and verify quality is maintained
5. Adjust threshold based on your specific needs

For more on effective context management, see [Effective Context Engineering for AI Agents](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents).